In [ ]:
%%capture
!pip install -r requirements.txt
from utils import *

wd = WarpDrive()

dataset_type = wd.get_args("dataset_type")


In [ ]:
## Helper functions
def read_parquet_from_azure(container, directory):
    with wd.store.get_object_readable_stream(container, directory) as stream:
        data = pd.read_parquet(stream)
    return data

def extract_windows(pargs):
    window_definitions = []
    for window in pargs.get("windows", {}).get("definition", []):
        for window_name, window_values in window.items():
            window_definitions.append({
                "window":     window_name,
                "start_date": window_values.get("start_date"),
                "end_date":   window_values.get("end_date")
            })
    return window_definitions

def add_segment_feature(df_features, new_feature_df, key='customer_id'):
    return df_features.merge(new_feature_df, on=key, how='left')

def safe_div(num, den):
    den_safe = den.mask(den == 0, pd.NA)
    return num / den_safe

def save_df_to_azure(data, container_name, file_path, output_type='parquet'):
    buffer = io.BytesIO()
    if output_type == 'parquet':
        data.to_parquet(buffer, index=False)
    buffer.seek(0)
    wd.store.put_object(container_name=container_name, key=file_path, data=buffer)
    print(f"Saved → {container_name}/{file_path}")


In [ ]:
import json
pargs = json.loads(wd.store.read_object(
    'nimbus-uno-usbank',
    'anomaly_detection/retail_banking/arguments/pipeline_arguments.json'
).decode('utf-8'))

container        = pargs['paths']['container']
prefix_base      = pargs['paths']['prefix_base']
total_num_windows = pargs['windows']['count']
start_date       = pargs['data_period']['start_date']
end_date         = pargs['data_period']['end_date']
windows_info     = extract_windows(pargs)

file_path        = 'anomaly_detection/retail_banking_new/data'
date_folder_fmt  = '_'.join([start_date.replace('-', ''), end_date.replace('-', '')])


In [ ]:
import os 
# Load full merged data for historical baseline features
tms_data_full = read_parquet_from_azure(
    container,
    os.path.join(file_path, date_folder_fmt, 'tms_data_full.parquet')
)

In [ ]:
tms_data_full['is_pep'].value_counts()

In [ ]:
CASH_CHANNELS    = ['Branch', 'Agent']
ATM_CHANNELS     = ['ATM']
DIGITAL_CHANNELS = ['Online Banking', 'Mobile App']
ACH_CHANNELS     = ['ACH']
WIRE_CHANNELS    = ['Wire']
CARD_CHANNELS    = ['POS']

In [ ]:
def build_segmentation_features_for_window(df_window, df_full, window_start, window_end):
    """
    Builds all segmentation features for a single rolling window.

    Parameters:
        df_window    : pre-filtered tms data for this window
        df_full      : complete tms data (all dates) — for historical baseline only
        window_start : window start date (pd.Timestamp)
        window_end   : window end date (pd.Timestamp)

    Returns:
        df_features  : customer-level feature dataframe for this window
    """

    df_temp          = df_window.copy()
    observation_days = (window_end - window_start).days

    df_temp['txn_month']     = df_temp['transaction_datetime'].dt.to_period('M')
    df_temp['txn_date']      = df_temp['transaction_datetime'].dt.date
    df_temp['TXN_DATE_ONLY'] = df_temp['transaction_datetime'].dt.date

    # ─────────────────────────────────────────────────────────────────────
    # SECTION 1 — AGGREGATE FEATURES
    # ─────────────────────────────────────────────────────────────────────

    # Total transaction count
    df_features = (
        df_temp.groupby('customer_id', as_index=False)['transaction_id']
        .count()
        .rename(columns={'transaction_id': 'txn_count'})
    )

    # Credit transaction count
    credit_txn_count = (
        df_temp.loc[df_temp['debit_credit_indicator'] == 'Credit']
        .groupby('customer_id', as_index=False)['transaction_id']
        .count()
        .rename(columns={'transaction_id': 'credit_txn_count'})
    )
    df_features = add_segment_feature(df_features, credit_txn_count)

    # Debit transaction count
    debit_txn_count = (
        df_temp.loc[df_temp['debit_credit_indicator'] == 'Debit']
        .groupby('customer_id', as_index=False)['transaction_id']
        .count()
        .rename(columns={'transaction_id': 'debit_txn_count'})
    )
    df_features = add_segment_feature(df_features, debit_txn_count)

    # Total credit amount
    credit = (
        df_temp.loc[df_temp['debit_credit_indicator'] == 'Credit']
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'total_credit'})
    )
    df_features = add_segment_feature(df_features, credit)

    # Total debit amount
    debit = (
        df_temp.loc[df_temp['debit_credit_indicator'] == 'Debit']
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'total_debit'})
    )
    df_features = add_segment_feature(df_features, debit)

    # Cash deposits (Branch + Agent, CREDIT)
    cash_total_credit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Credit') &
            (df_temp['transaction_channel'].isin(CASH_CHANNELS))
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'cash_total_credit'})
    )
    df_features = add_segment_feature(df_features, cash_total_credit)

    # Cash withdrawals (Branch + Agent, DEBIT)
    cash_total_debit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['transaction_channel'].isin(CASH_CHANNELS))
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'cash_total_debit'})
    )
    df_features = add_segment_feature(df_features, cash_total_debit)

    # Cash transaction count
    cash_txn_count = (
        df_temp.loc[df_temp['transaction_channel'].isin(CASH_CHANNELS)]
        .groupby('customer_id', as_index=False)['transaction_id']
        .count()
        .rename(columns={'transaction_id': 'cash_txn_count'})
    )
    df_features = add_segment_feature(df_features, cash_txn_count)

    # Days with multiple cash transactions
    cash_txns = df_temp.loc[df_temp['transaction_channel'].isin(CASH_CHANNELS)].copy()
    cash_txn_daily_counts = (
        cash_txns.groupby(['customer_id', 'txn_date']).size().reset_index(name='txn_count')
    )
    multiple_cash_same_day_count = (
        cash_txn_daily_counts.loc[cash_txn_daily_counts['txn_count'] > 1]
        .groupby('customer_id', as_index=False).size()
        .rename(columns={'size': 'multiple_cash_same_day_count'})
    )
    df_features = add_segment_feature(df_features, multiple_cash_same_day_count)

    # ATM total withdrawal
    atm_total_debit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['transaction_channel'].isin(ATM_CHANNELS))
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'atm_total_debit'})
    )
    df_features = add_segment_feature(df_features, atm_total_debit)

    # ATM transaction count
    atm_txn_count = (
        df_temp.loc[df_temp['transaction_channel'].isin(ATM_CHANNELS)]
        .groupby('customer_id').size().rename('atm_txn_count')
    )
    df_features = add_segment_feature(df_features, atm_txn_count)

    # Card total spend (POS)
    card_total_spend = (
        df_temp.loc[df_temp['transaction_channel'].isin(CARD_CHANNELS)]
        .groupby('customer_id')['transaction_amount_usd']
        .sum().rename('card_total_spend')
    )
    df_features = add_segment_feature(df_features, card_total_spend)

    # Card transaction count
    card_txn_count = (
        df_temp.loc[df_temp['transaction_channel'].isin(CARD_CHANNELS)]
        .groupby('customer_id').size().rename('card_txn_count')
    )
    df_features = add_segment_feature(df_features, card_txn_count)

    # Digital total debit (Online Banking + Mobile App)
    digital_total_debit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['transaction_channel'].isin(DIGITAL_CHANNELS))
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'digital_total_debit'})
    )
    df_features = add_segment_feature(df_features, digital_total_debit)

    # Digital total credit
    digital_total_credit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Credit') &
            (df_temp['transaction_channel'].isin(DIGITAL_CHANNELS))
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'digital_total_credit'})
    )
    df_features = add_segment_feature(df_features, digital_total_credit)

    # Digital credit count
    digital_txn_credit_count = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Credit') &
            (df_temp['transaction_channel'].isin(DIGITAL_CHANNELS))
        ]
        .groupby('customer_id').size().rename('digital_txn_credit_count')
    )
    df_features = add_segment_feature(df_features, digital_txn_credit_count)

    # Digital debit count
    digital_txn_debit_count = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['transaction_channel'].isin(DIGITAL_CHANNELS))
        ]
        .groupby('customer_id').size().rename('digital_txn_debit_count')
    )
    df_features = add_segment_feature(df_features, digital_txn_debit_count)

    # Unique channels used
    unique_channels_used = (
        df_temp.groupby('customer_id')['transaction_channel']
        .nunique().rename('unique_channels_used')
    )
    df_features = add_segment_feature(df_features, unique_channels_used)

    # ACH credit (replaces cheque credit)
    ach_total_credit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Credit') &
            (df_temp['transaction_channel'].isin(ACH_CHANNELS))
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'ach_total_credit'})
    )
    df_features = add_segment_feature(df_features, ach_total_credit)

    # ACH debit
    ach_total_debit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['transaction_channel'].isin(ACH_CHANNELS))
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'ach_total_debit'})
    )
    df_features = add_segment_feature(df_features, ach_total_debit)

    # ACH credit count
    ach_txn_credit_count = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Credit') &
            (df_temp['transaction_channel'].isin(ACH_CHANNELS))
        ]
        .groupby('customer_id').size().rename('ach_txn_credit_count')
    )
    df_features = add_segment_feature(df_features, ach_txn_credit_count)

    # ACH debit count
    ach_txn_debit_count = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['transaction_channel'].isin(ACH_CHANNELS))
        ]
        .groupby('customer_id').size().rename('ach_txn_debit_count')
    )
    df_features = add_segment_feature(df_features, ach_txn_debit_count)

    # Wire credit
    wire_total_credit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Credit') &
            (df_temp['transaction_channel'].isin(WIRE_CHANNELS))
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'wire_total_credit'})
    )
    df_features = add_segment_feature(df_features, wire_total_credit)

    # Wire debit
    wire_total_debit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['transaction_channel'].isin(WIRE_CHANNELS))
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'wire_total_debit'})
    )
    df_features = add_segment_feature(df_features, wire_total_debit)

    # Wire credit count
    wire_txn_credit_count = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Credit') &
            (df_temp['transaction_channel'].isin(WIRE_CHANNELS))
        ]
        .groupby('customer_id').size().rename('wire_txn_credit_count')
    )
    df_features = add_segment_feature(df_features, wire_txn_credit_count)

    # Wire debit count
    wire_txn_debit_count = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['transaction_channel'].isin(WIRE_CHANNELS))
        ]
        .groupby('customer_id').size().rename('wire_txn_debit_count')
    )
    df_features = add_segment_feature(df_features, wire_txn_debit_count)

    # Cross-border credit (using is_international flag)
    cross_border_total_credit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Credit') &
            (df_temp['is_international'] == True)
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'cross_border_total_credit'})
    )
    df_features = add_segment_feature(df_features, cross_border_total_credit)

    # Cross-border debit
    cross_border_total_debit = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['is_international'] == True)
        ]
        .groupby('customer_id', as_index=False)['transaction_amount_usd']
        .sum()
        .rename(columns={'transaction_amount_usd': 'cross_border_total_debit'})
    )
    df_features = add_segment_feature(df_features, cross_border_total_debit)

    # Cross-border credit count
    cross_border_txn_credit_count = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Credit') &
            (df_temp['is_international'] == True)
        ]
        .groupby('customer_id').size().rename('cross_border_txn_credit_count')
    )
    df_features = add_segment_feature(df_features, cross_border_txn_credit_count)

    # Cross-border debit count
    cross_border_txn_debit_count = (
        df_temp.loc[
            (df_temp['debit_credit_indicator'] == 'Debit') &
            (df_temp['is_international'] == True)
        ]
        .groupby('customer_id').size().rename('cross_border_txn_debit_count')
    )
    df_features = add_segment_feature(df_features, cross_border_txn_debit_count)

    # Unique foreign countries
    foreign_country_count = (
        df_temp.loc[df_temp['is_international'] == True]
        .groupby('customer_id', as_index=False)['counterparty_country']
        .nunique()
        .rename(columns={'counterparty_country': 'foreign_country_count'})
    )
    df_features = add_segment_feature(df_features, foreign_country_count)

    # High-risk country transaction count (using precomputed flag)
    high_risk_country_txn_count = (
        df_temp.loc[df_temp['is_high_risk_country'] == True]
        .groupby('customer_id', as_index=False).size()
        .rename(columns={'size': 'high_risk_country_txn_count'})
    )
    df_features = add_segment_feature(df_features, high_risk_country_txn_count)

    # Active days count
    active_days_count = (
        df_temp.groupby('customer_id')['TXN_DATE_ONLY']
        .nunique()
        .reset_index(name='active_days_count')
    )
    df_features = add_segment_feature(df_features, active_days_count)

    # Inactive days count
    inactive_days_count = (
        active_days_count.assign(
            inactive_days_count=lambda x: observation_days - x['active_days_count']
        )[['customer_id', 'inactive_days_count']]
    )
    df_features = add_segment_feature(df_features, inactive_days_count)

    # Weekend transaction count
    weekend_txn_count = (
        df_temp.loc[df_temp['transaction_datetime'].dt.weekday.isin([5, 6])]
        .groupby('customer_id').size().rename('weekend_txn_count')
    )
    df_features = add_segment_feature(df_features, weekend_txn_count)

    # Night transaction count (9pm - 5am)
    night_txn_count = (
        df_temp.loc[
            (df_temp['transaction_datetime'].dt.hour >= 21) |
            (df_temp['transaction_datetime'].dt.hour < 5)
        ]
        .groupby('customer_id').size().rename('night_txn_count')
    )
    df_features = add_segment_feature(df_features, night_txn_count)

    # Unique counterparties
    unique_counterparty_count = (
        df_temp.loc[df_temp['counterparty_account'].notna()]
        .groupby('customer_id')['counterparty_account']
        .nunique()
        .reset_index(name='unique_counterparty_count')
    )
    df_features = add_segment_feature(df_features, unique_counterparty_count)

    # New counterparties — historical baseline is everything BEFORE this window
    historical = (
        df_full.loc[
            (df_full['transaction_datetime'] < window_start) &
            (df_full['counterparty_account'].notna())
        ]
        .groupby('customer_id')['counterparty_account'].apply(set)
    )
    recent = (
        df_temp.loc[df_temp['counterparty_account'].notna()]
        .groupby('customer_id')['counterparty_account'].apply(set)
    )
    new_counterparty_count = (
        recent.combine(
            historical,
            lambda r, h: (
                len(r - h) if isinstance(r, set) and isinstance(h, set)
                else len(r) if isinstance(r, set)
                else 0
            )
        )
        .rename('new_counterparty_count')
        .reset_index()
    )
    df_features = add_segment_feature(df_features, new_counterparty_count)

    # Repeated counterparties
    repeated_counterparty_count = (
        df_temp.loc[df_temp['counterparty_account'].notna()]
        .groupby(['customer_id', 'counterparty_account']).size()
        .reset_index(name='txn_count')
        .loc[lambda x: x['txn_count'] > 1]
        .groupby('customer_id').size()
        .reset_index(name='repeated_counterparty_count')
    )
    df_features = add_segment_feature(df_features, repeated_counterparty_count)

    # Top counterparty transaction count
    top_counterparty_txn_count = (
        df_temp.loc[df_temp['counterparty_account'].notna()]
        .groupby(['customer_id', 'counterparty_account']).size()
        .reset_index(name='txn_count')
        .groupby('customer_id')['txn_count'].max()
        .reset_index(name='top_counterparty_txn_count')
    )
    df_features = add_segment_feature(df_features, top_counterparty_txn_count)

    # Credit-debit amount gap
    credit_debit_gap = (
        df_temp.assign(
            credit_amt=lambda x: x['transaction_amount_usd'].where(x['debit_credit_indicator'] == 'Credit', 0),
            debit_amt =lambda x: x['transaction_amount_usd'].where(x['debit_credit_indicator'] == 'Debit',  0)
        )
        .groupby('customer_id', as_index=False)
        .agg(total_credit=('credit_amt', 'sum'), total_debit=('debit_amt', 'sum'))
    )
    credit_debit_gap['credit_debit_amount_gap'] = (
        credit_debit_gap['total_credit'] - credit_debit_gap['total_debit']
    )
    df_features = add_segment_feature(df_features, credit_debit_gap[['customer_id', 'credit_debit_amount_gap']])

    # Months where credits exceed debits
    monthly_cd = (
        df_temp.groupby(['customer_id', 'txn_month', 'debit_credit_indicator'])['transaction_amount_usd']
        .sum().unstack(fill_value=0).reset_index()
    )
    months_credit_gt_debit = (
        monthly_cd.assign(flag=lambda x: x.get('Credit', 0) > x.get('Debit', 0))
        .groupby('customer_id', as_index=False)['flag'].sum()
        .rename(columns={'flag': 'months_credit_gt_debit_count'})
    )
    df_features = add_segment_feature(df_features, months_credit_gt_debit)

    # Months where debits exceed credits
    months_debit_gt_credit = (
        monthly_cd.assign(flag=lambda x: x.get('Debit', 0) > x.get('Credit', 0))
        .groupby('customer_id', as_index=False)['flag'].sum()
        .rename(columns={'flag': 'months_debit_gt_credit_count'})
    )
    df_features = add_segment_feature(df_features, months_debit_gt_credit)

    # High-risk channel transaction count (Cash + Wire + International)
    high_risk_channel_txn_count = (
        df_temp.loc[
            (df_temp['transaction_channel'].isin(CASH_CHANNELS + WIRE_CHANNELS)) |
            (df_temp['is_international'] == True)
        ]
        .groupby('customer_id').size().rename('high_risk_channel_txn_count')
    )
    df_features = add_segment_feature(df_features, high_risk_channel_txn_count)

    # PEP-related transactions
    pep_related_txn_count = (
        df_temp.loc[df_temp['is_pep'] == 'Yes']
        .groupby('customer_id').size().rename('pep_related_txn_count')
    )
    df_features = add_segment_feature(df_features, pep_related_txn_count)

    # Rapid activity post onboarding — uses full history not window data
    early_days = 30
    rapid_activity_post_onboarding_count = (
        df_full.loc[
            (df_full['transaction_datetime'] >= df_full['onboarding_date']) &
            (df_full['transaction_datetime'] <= df_full['onboarding_date'] + pd.Timedelta(days=early_days))
        ]
        .groupby('customer_id').size().rename('rapid_activity_post_onboarding_count')
    )
    df_features = add_segment_feature(df_features, rapid_activity_post_onboarding_count)

    # ─────────────────────────────────────────────────────────────────────
    # SECTION 2 — RATIO FEATURES
    # ─────────────────────────────────────────────────────────────────────

    base_agg = (
        df_temp.groupby('customer_id').apply(lambda x: pd.Series({
            'total_txn_amount':    x['transaction_amount_usd'].sum(),
            'total_txn_count':     len(x),
            'total_credit_amount': x.loc[x['debit_credit_indicator'] == 'Credit', 'transaction_amount_usd'].sum(),
            'total_debit_amount':  x.loc[x['debit_credit_indicator'] == 'Debit',  'transaction_amount_usd'].sum(),
            'credit_txn_count':    (x['debit_credit_indicator'] == 'Credit').sum(),
            'debit_txn_count':     (x['debit_credit_indicator'] == 'Debit').sum(),
        })).reset_index()
    )

    def channel_agg(tag, condition):
        return (
            df_temp.loc[condition]
            .groupby('customer_id')
            .agg(
                amount=('transaction_amount_usd', 'sum'),
                count =('transaction_amount_usd', 'count')
            )
            .rename(columns={'amount': f'{tag}_amount', 'count': f'{tag}_count'})
        )

    cash_ch         = channel_agg('cash',         df_temp['transaction_channel'].isin(CASH_CHANNELS))
    card_ch         = channel_agg('card',          df_temp['transaction_channel'].isin(CARD_CHANNELS))
    digital_ch      = channel_agg('digital',       df_temp['transaction_channel'].isin(DIGITAL_CHANNELS))
    ach_ch          = channel_agg('ach',           df_temp['transaction_channel'].isin(ACH_CHANNELS))
    wire_ch         = channel_agg('wire',          df_temp['transaction_channel'].isin(WIRE_CHANNELS))
    cross_border_ch = channel_agg('cross_border',  df_temp['is_international'] == True)

    ratio_base = base_agg.copy()
    for df_ch in [cash_ch, card_ch, digital_ch, ach_ch, wire_ch, cross_border_ch]:
        ratio_base = ratio_base.merge(df_ch, on='customer_id', how='left')
    ratio_base.fillna(0, inplace=True)

    df_ratio_features = ratio_base.assign(
        credit_debit_ratio           = safe_div(ratio_base['total_credit_amount'], ratio_base['total_debit_amount']),
        credit_debit_txn_count_ratio = safe_div(ratio_base['credit_txn_count'],    ratio_base['debit_txn_count']),
        cash_usage_ratio             = safe_div(ratio_base['cash_amount'],         ratio_base['total_txn_amount']),
        cash_txn_count_ratio         = safe_div(ratio_base['cash_count'],          ratio_base['total_txn_count']),
        card_usage_ratio             = safe_div(ratio_base['card_amount'],         ratio_base['total_debit_amount']),
        card_txn_count_ratio         = safe_div(ratio_base['card_count'],          ratio_base['total_txn_count']),
        digital_usage_ratio          = safe_div(ratio_base['digital_amount'],      ratio_base['total_txn_amount']),
        digital_txn_count_ratio      = safe_div(ratio_base['digital_count'],       ratio_base['total_txn_count']),
        ach_usage_ratio              = safe_div(ratio_base['ach_amount'],          ratio_base['total_txn_amount']),
        ach_txn_count_ratio          = safe_div(ratio_base['ach_count'],           ratio_base['total_txn_count']),
        wire_usage_ratio             = safe_div(ratio_base['wire_amount'],         ratio_base['total_txn_amount']),
        wire_txn_count_ratio         = safe_div(ratio_base['wire_count'],          ratio_base['total_txn_count']),
        cross_border_ratio           = safe_div(ratio_base['cross_border_amount'], ratio_base['total_txn_amount']),
        cross_border_txn_count_ratio = safe_div(ratio_base['cross_border_count'],  ratio_base['total_txn_count']),
    )[[
        'customer_id',
        'credit_debit_ratio', 'credit_debit_txn_count_ratio',
        'cash_usage_ratio', 'cash_txn_count_ratio',
        'card_usage_ratio', 'card_txn_count_ratio',
        'digital_usage_ratio', 'digital_txn_count_ratio',
        'ach_usage_ratio', 'ach_txn_count_ratio',
        'wire_usage_ratio', 'wire_txn_count_ratio',
        'cross_border_ratio', 'cross_border_txn_count_ratio'
    ]]
    df_features = add_segment_feature(df_features, df_ratio_features)

    # High-risk country ratios (using precomputed flag)
    high_risk_agg = (
        df_temp.loc[df_temp['is_high_risk_country'] == True]
        .groupby('customer_id')
        .agg(
            high_risk_country_amount   =('transaction_amount_usd', 'sum'),
            high_risk_country_txn_count=('transaction_amount_usd', 'count')
        )
        .reset_index()
    )
    ratio_base = ratio_base.merge(high_risk_agg, on='customer_id', how='left').fillna(0)
    ratio_base['high_risk_country_ratio']           = safe_div(ratio_base['high_risk_country_amount'],    ratio_base['total_txn_amount'])
    ratio_base['high_risk_country_txn_count_ratio'] = safe_div(ratio_base['high_risk_country_txn_count'], ratio_base['total_txn_count'])
    df_features = add_segment_feature(df_features, ratio_base[['customer_id', 'high_risk_country_ratio', 'high_risk_country_txn_count_ratio']])

    # Round amount ratio (using precomputed flag)
    round_txn_count = (
        df_temp.loc[df_temp['is_round_amount'] == True]
        .groupby('customer_id').size().reset_index(name='round_txn_count')
    )
    total_txn_count = df_temp.groupby('customer_id').size().reset_index(name='txn_count')
    round_ratio_df  = round_txn_count.merge(total_txn_count, on='customer_id', how='left')
    round_ratio_df['round_amount_ratio'] = round_ratio_df['round_txn_count'] / round_ratio_df['txn_count']
    round_ratio_df  = round_ratio_df[['customer_id', 'round_amount_ratio']]

    # ATM withdrawal ratio
    atm_txn_count_r = (
        df_temp.loc[
            (df_temp['transaction_channel'].isin(ATM_CHANNELS)) &
            (df_temp['debit_credit_indicator'] == 'Debit')
        ]
        .groupby('customer_id').size().reset_index(name='atm_txn_count')
    )
    debit_txn_count_r = (
        df_temp.loc[df_temp['debit_credit_indicator'] == 'Debit']
        .groupby('customer_id').size().reset_index(name='debit_txn_count')
    )
    atm_ratio_df = atm_txn_count_r.merge(debit_txn_count_r, on='customer_id', how='left')
    atm_ratio_df['atm_withdrawal_count_ratio'] = atm_ratio_df['atm_txn_count'] / atm_ratio_df['debit_txn_count']
    atm_ratio_df = atm_ratio_df[['customer_id', 'atm_withdrawal_count_ratio']]

    # Active / inactive days ratio
    active_days_df = (
        df_temp.groupby('customer_id')['TXN_DATE_ONLY']
        .nunique().reset_index(name='active_days_count')
    )
    active_days_df['inactive_days_count'] = observation_days - active_days_df['active_days_count']
    active_days_df['active_days_ratio']   = active_days_df['active_days_count'] / observation_days
    active_days_df['inactive_days_ratio'] = active_days_df['inactive_days_count'] / observation_days
    activity_ratio_df = active_days_df[['customer_id', 'active_days_ratio', 'inactive_days_ratio']]

    for df_new in [round_ratio_df, atm_ratio_df, activity_ratio_df]:
        df_features = df_features.merge(df_new, on='customer_id', how='left')

    # ─────────────────────────────────────────────────────────────────────
    # SECTION 3 — EWMA FEATURES
    # ─────────────────────────────────────────────────────────────────────

    df_temp['txn_month'] = df_temp['transaction_datetime'].dt.to_period('M')

    monthly = (
        df_temp.groupby(['customer_id', 'txn_month'])
        .apply(lambda x: pd.Series({
            'credit_amt':          x.loc[x['debit_credit_indicator'] == 'Credit', 'transaction_amount_usd'].sum(),
            'debit_amt':           x.loc[x['debit_credit_indicator'] == 'Debit',  'transaction_amount_usd'].sum(),
            'cash_credit':         x.loc[(x['transaction_channel'].isin(CASH_CHANNELS))    & (x['debit_credit_indicator'] == 'Credit'), 'transaction_amount_usd'].sum(),
            'cash_debit':          x.loc[(x['transaction_channel'].isin(CASH_CHANNELS))    & (x['debit_credit_indicator'] == 'Debit'),  'transaction_amount_usd'].sum(),
            'card_credit':         x.loc[(x['transaction_channel'].isin(CARD_CHANNELS))    & (x['debit_credit_indicator'] == 'Credit'), 'transaction_amount_usd'].sum(),
            'card_debit':          x.loc[(x['transaction_channel'].isin(CARD_CHANNELS))    & (x['debit_credit_indicator'] == 'Debit'),  'transaction_amount_usd'].sum(),
            'wire_credit':         x.loc[(x['transaction_channel'].isin(WIRE_CHANNELS))    & (x['debit_credit_indicator'] == 'Credit'), 'transaction_amount_usd'].sum(),
            'wire_debit':          x.loc[(x['transaction_channel'].isin(WIRE_CHANNELS))    & (x['debit_credit_indicator'] == 'Debit'),  'transaction_amount_usd'].sum(),
            'digital_credit':      x.loc[(x['transaction_channel'].isin(DIGITAL_CHANNELS)) & (x['debit_credit_indicator'] == 'Credit'), 'transaction_amount_usd'].sum(),
            'digital_debit':       x.loc[(x['transaction_channel'].isin(DIGITAL_CHANNELS)) & (x['debit_credit_indicator'] == 'Debit'),  'transaction_amount_usd'].sum(),
            'ach_credit':          x.loc[(x['transaction_channel'].isin(ACH_CHANNELS))     & (x['debit_credit_indicator'] == 'Credit'), 'transaction_amount_usd'].sum(),
            'ach_debit':           x.loc[(x['transaction_channel'].isin(ACH_CHANNELS))     & (x['debit_credit_indicator'] == 'Debit'),  'transaction_amount_usd'].sum(),
            'cross_border_credit': x.loc[(x['is_international'] == True)                   & (x['debit_credit_indicator'] == 'Credit'), 'transaction_amount_usd'].sum(),
            'cross_border_debit':  x.loc[(x['is_international'] == True)                   & (x['debit_credit_indicator'] == 'Debit'),  'transaction_amount_usd'].sum(),
        }))
        .reset_index()
    )

    ewma_df = (
        monthly.sort_values(['customer_id', 'txn_month'])
        .groupby('customer_id')
        .apply(lambda x: pd.Series({
            'credit_expavg':              x['credit_amt'].ewm(span=3).mean().iloc[-1],
            'debit_expavg':               x['debit_amt'].ewm(span=3).mean().iloc[-1],
            'cash_expavg_credit':         x['cash_credit'].ewm(span=3).mean().iloc[-1],
            'cash_expavg_debit':          x['cash_debit'].ewm(span=3).mean().iloc[-1],
            'card_expavg_credit':         x['card_credit'].ewm(span=3).mean().iloc[-1],
            'card_expavg_debit':          x['card_debit'].ewm(span=3).mean().iloc[-1],
            'wire_expavg_credit':         x['wire_credit'].ewm(span=3).mean().iloc[-1],
            'wire_expavg_debit':          x['wire_debit'].ewm(span=3).mean().iloc[-1],
            'digital_expavg_credit':      x['digital_credit'].ewm(span=3).mean().iloc[-1],
            'digital_expavg_debit':       x['digital_debit'].ewm(span=3).mean().iloc[-1],
            'ach_expavg_credit':          x['ach_credit'].ewm(span=3).mean().iloc[-1],
            'ach_expavg_debit':           x['ach_debit'].ewm(span=3).mean().iloc[-1],
            'cross_border_expavg_credit': x['cross_border_credit'].ewm(span=3).mean().iloc[-1],
            'cross_border_expavg_debit':  x['cross_border_debit'].ewm(span=3).mean().iloc[-1],
        }))
        .reset_index()
    )

    df_features = add_segment_feature(df_features, ewma_df)
    df_features = df_features.fillna(0)

    return df_features



In [ ]:
# ─────────────────────────────────────────────
# TRAIN: Rolling Window Feature Engineering
# ─────────────────────────────────────────────

if dataset_type == 'train':

    window_feature_dfs = {}

    for window in windows_info:
        window_name  = window["window"]
        window_start = pd.to_datetime(window["start_date"])
        window_end   = pd.to_datetime(window["end_date"])

        print(f"Building features for {window_name}: {window_start.date()} → {window_end.date()}")

        df_window = read_parquet_from_azure(
            container,
            os.path.join(file_path, date_folder_fmt, f'tms_data_{window_name}.parquet')
        )

        df_window_features = build_segmentation_features_for_window(
            df_window, tms_data_full, window_start, window_end
        )

        df_window_features['window'] = window_name
        window_feature_dfs[window_name] = df_window_features

        print(f"  → {df_window_features.shape[0]} customers, {df_window_features.shape[1]} features")

    # ── Combine by averaging across windows ──────────────────────────────

    feature_cols = [
        c for c in list(window_feature_dfs.values())[0].columns
        if c not in ['customer_id', 'window']
    ]

    all_windows_stacked = pd.concat(window_feature_dfs.values(), ignore_index=True)

    df_features_combined = (
        all_windows_stacked
        .groupby('customer_id')[feature_cols]
        .mean()
        .reset_index()
    )

    # Window presence count as a feature
    window_presence = (
        all_windows_stacked.groupby('customer_id')['window']
        .nunique()
        .reset_index(name='window_count')
    )
    df_features_combined = df_features_combined.merge(window_presence, on='customer_id', how='left')

    print(f"\nCombined train features: {df_features_combined.shape[0]} customers × {df_features_combined.shape[1]} features")

    # ── Window-level summary ──────────────────────────────────────────────

    summary_rows = []
    for window_name, df_wf in window_feature_dfs.items():
        w = next(w for w in windows_info if w["window"] == window_name)
        summary_rows.append({
            "Window":     window_name,
            "Start Date": w["start_date"],
            "End Date":   w["end_date"],
            "#Customers": df_wf.shape[0],
            "#Features":  df_wf.shape[1] - 2
        })
    wd.save_table(pd.DataFrame(summary_rows), f"Segmentation Feature Engineering Window Summary")

    # ── Save ─────────────────────────────────────────────────────────────

    save_df_to_azure(
        df_features_combined, container,
        os.path.join(file_path, date_folder_fmt, 'segmentation_features_train.parquet')
    )
    wd.save_table(df_features_combined, f"Segmentation Features - Train")
    del all_windows_stacked, window_feature_dfs

# ─────────────────────────────────────────────
# TEST: Single Window Feature Engineering
# ─────────────────────────────────────────────

else:

    training_end_date = pd.to_datetime(windows_info[-1]["end_date"])
    test_start        = training_end_date + pd.Timedelta(days=1)
    test_end          = pd.to_datetime(end_date)

    print(f"Building features for test: {test_start.date()} → {test_end.date()}")

    df_test = read_parquet_from_azure(
    container,
    os.path.join(file_path, date_folder_fmt, 'tms_data_test.parquet')
    )

    df_features_test = build_segmentation_features_for_window(
        df_test, tms_data_full, test_start, test_end
    )

    print(f"  → {df_features_test.shape[0]} customers, {df_features_test.shape[1]} features")

    save_df_to_azure(
        df_features_test, container,
        os.path.join(file_path, date_folder_fmt, 'segmentation_features_test.parquet')
    )
    wd.save_table(df_features_test, f"Segmentation Features - Test")
    del df_features_test
